# 4. A two-belt system: SED, fit and images

Many debris discs have an inner warm belt and an outer cold belt. `artefact` models this with `dtype="multi"` and a list of belts that share one grain size distribution. Each belt has its own shape and an optional relative `mass`.

This notebook:

1. builds a warm belt (5 au) plus cold belt (80 au) around an F star at 30 pc,
2. separates the SED into the emission of each belt,
3. fits synthetic photometry to recover the system,
4. images the best-fit model at wavelengths that pick out each belt.

In [ ]:
import sys
from pathlib import Path

# The notebooks live in notebooks/, the helpers in example/
sys.path.insert(0, str(Path.cwd().parent / "example"))

import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

In [ ]:
import lmfit
from example_utils import (
    belt_sed, model_at, plot_sed, run_model, show_image, two_belt_parameters,
)

## The model

Each entry in `belts` takes the same parameters as a single belt (`dtype`, `rpeak`/`rfwhm` for `"gauss"`, `rin`/`rout`/`alpha_out` for `"onepl"` and so on), plus:

- `nring`: annuli in that belt,
- `mass`: relative share of the total dust mass (normalised across belts).

The annuli are the union of every belt's edges, so a gap between belts becomes one empty annulus.

In [ ]:
TRUTH = dict(log_mdust=-1.3, q=-3.5, warm_fraction=0.02, rpeak_warm=5.0, rpeak_cold=80.0)
parameters = two_belt_parameters(**TRUTH)
parameters["belts"]

In [ ]:
model = run_model(parameters)
print(f"{len(model.radii)} annuli; belt_scale shape {model.belt_scale.shape}")

fig, (ax_sed, ax_disc) = plt.subplots(1, 2, figsize=(13, 5))
plot_sed(model, ax=ax_sed, belts=True, title="Two-belt SED")
ax_disc.plot(model.radii, model.belt_scale[0], label="warm belt")
ax_disc.plot(model.radii, model.belt_scale[1], label="cold belt")
ax_disc.set_xscale("log")
ax_disc.set_xlabel("Radius (au)")
ax_disc.set_ylabel("Fraction of all grains per annulus")
ax_disc.legend()
plt.show()

The warm belt dominates the mid-infrared (10-30 um) and the cold belt the far-infrared and sub-mm. A single belt cannot reproduce both, which is the signature of two components in the SED.

## Synthetic photometry

To test the fit we sample the true model at typical photometric bands and add 5% Gaussian noise (seeded for reproducibility).

In [ ]:
WAVELENGTHS = np.array([0.44, 0.55, 0.79, 1.25, 1.65, 2.2, 3.4, 4.6, 12.0, 22.0, 24.0,
                        70.0, 100.0, 160.0, 450.0, 870.0, 1300.0])

truth = run_model(parameters, emission_mode="bb")
rng = np.random.default_rng(42)
flux_true = model_at(truth, WAVELENGTHS)
unc = 0.05 * flux_true
flux = flux_true + rng.normal(0.0, unc)

## Fit

Five parameters: total dust mass, size slope, the warm belt's mass fraction and the two belt radii. The star is assumed known, so only points beyond 8 um (the infrared excess) are fitted. This takes a few minutes.

In [ ]:
fit_mask = WAVELENGTHS > 8.0

params = lmfit.Parameters()
params.add("log_mdust", value=-1.0, min=-4.0, max=1.0)
params.add("q", value=-3.6, min=-4.5, max=-3.0)
params.add("warm_fraction", value=0.05, min=1e-4, max=0.5)
params.add("rpeak_warm", value=4.0, min=2.0, max=15.0)   # keeps rpeak - 3 sigma > 0
params.add("rpeak_cold", value=70.0, min=40.0, max=150.0)


def residual(theta):
    m = run_model(two_belt_parameters(**theta.valuesdict()), emission_mode="bb")
    return (model_at(m, WAVELENGTHS[fit_mask]) - flux[fit_mask]) / unc[fit_mask]


result = lmfit.minimize(residual, params, method="least_squares")
print(lmfit.fit_report(result))

In [ ]:
for name, true_value in TRUTH.items():
    fitted = result.params[name]
    print(f"{name:>14}: fit {fitted.value:9.3g} +/- {fitted.stderr or float('nan'):.2g}   true {true_value:9.3g}")

In [ ]:
best = run_model(two_belt_parameters(**result.params.valuesdict()))
best.obs_wave, best.obs_flux, best.obs_uncs = WAVELENGTHS, flux, unc

fig, ax = plt.subplots(figsize=(7, 5))
plot_sed(best, ax=ax, belts=True, title="Two-belt fit")
plt.show()

## Images

At 30 pc the warm belt is about 0.3 arcsec across and the cold belt several arcsec, so no single field of view shows both well. We zoom in with a fine `pixscale` for the 24 um image of the warm belt.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
show_image(
    best.make_image(24.0, component="emit", inc=50, pa=20, npix=128, pixscale=0.01, beam=0.04),
    ax=axes[0], title="24 um, zoomed: warm belt",
)
show_image(best.make_image(70.0, component="emit", inc=50, pa=20, beam=1.0),
           ax=axes[1], title="70 um: both belts")
show_image(best.make_image(870.0, component="emit", inc=50, pa=20, beam=0.5),
           ax=axes[2], title="870 um: cold belt")
plt.show()

## Where does each belt's light come from?

`belt_sed` splits the thermal emission of the best fit by belt, using `belt_scale` (each belt's share of the grains in each annulus).

In [ ]:
warm, cold = belt_sed(best)
for wavelength in (24.0, 70.0, 870.0):
    i = np.argmin(np.abs(best.sed_wave - wavelength))
    print(f"{wavelength:6.0f} um: warm belt provides {warm[i] / (warm[i] + cold[i]):.0%} of the dust emission")